# EXP-2026-003: diagnose MSF conditioning on frozen weights

This notebook calls Python modules; it does not implement a model or training loop.
Read `experiments/registry/EXP-2026-003.md` for the detailed frozen plan.

- **Default `PHASE = "loss"`:** compare the author checkpoint and Stage5 r2 with own/shuffled/zero pre-op MRI/mask channels, at matched target latents, noise and text.
- **Optional `PHASE = "full"`:** also run the sampling factorial and guidance controls. Use a **new RUN_TAG**. Runtime has not been measured.
- **Zero training steps.** Keep Stage5 Part2 pending; this diagnostic is not its replacement.
- All text is **target-informed class text (oracle)**. Added residual-positive validation sentinels are explicitly oracle-selected scoring cases, separate from the frozen previews.
- Original MSF is a text-only control, not a patient-preserving baseline by design.
- Inputs are read from approved Drive storage and staged locally. Outputs are written locally, uploaded by Drive folder ID and confirmed by server MD5.
- Keep all outputs private. **Clear notebook outputs before committing/sharing.** No token should be pasted into this notebook or chat.

In [ ]:
REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/MedSegFactory.git"
BRANCH = "exp-2026-003-msf-diagnostics"
WORK = "/content/MedSegFactory-msf-diagnostics"
RUN_TAG = "diagnostic_r1"  # immutable; change after any attempt that created output
PHASE = "loss"  # "full" adds sampling; use a new RUN_TAG
DRIVE_RHUH = "/content/drive/MyDrive/RHUH-GBM"
DATA_ON_DRIVE = ("RHUH-GBM", "synthesis", "EXP-2026-001")
PARENT_RUN = ("stage5_runs", "r2")

## 1. Clone and install the diagnostic dependencies

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if not Path(WORK).exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK], check=True)
else:
    current = subprocess.check_output(["git", "-C", WORK, "branch", "--show-current"], text=True).strip()
    assert current == BRANCH, "This checkout belongs to another task; choose a different WORK"
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "diffusers==0.40.0", "transformers==5.17.0", "accelerate==1.15.0",
    "SimpleITK==2.5.6", "nibabel==5.4.2", "pydicom==3.0.2",
    "numpy==2.4.6", "scipy==1.16.3", "matplotlib>=3.9,<4", "pillow"], check=True)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

## 2. Read parent artifacts by Drive ID and verify local output storage
The preflight checks output access before model loading. Existing runs are never reused.

In [ ]:
import json
import shutil
from google.colab import auth, drive
from googleapiclient.discovery import build
from rhuh_synthesis.drive_api import DriveRun
from rhuh_synthesis.manifest import file_sha256

assert PHASE in ("loss", "full")
assert RUN_TAG and RUN_TAG not in (".", "..") and Path(RUN_TAG).name == RUN_TAG
config = json.loads(Path("configs/experiments/EXP-2026-003.json").read_text())
LOCAL_BASE = Path("/content/msf003_data")
DATASET = LOCAL_BASE / "EXP-2026-001"
OUTPUT = LOCAL_BASE / "EXP-2026-003" / RUN_TAG
PARENT = Path("/content/msf003_parent_r2")
assert not OUTPUT.exists(), "Choose a new RUN_TAG; previous attempts remain immutable"

drive.mount("/content/drive")
auth.authenticate_user()
api = build("drive", "v3", cache_discovery=False)
parent_drive = DriveRun.open(api, DATA_ON_DRIVE, PARENT_RUN, PARENT, create=False)
for name in ("record.json", "record.json.sha256", "train_log.jsonl"):
    parent_drive.fetch(name)
assert file_sha256(PARENT / "record.json") == config["parent_record_sha256"], "Unexpected parent record"
parent = json.loads((PARENT / "record.json").read_text())
TRAINED_WEIGHTS = parent_drive.fetch(parent["weights"]["file"])
assert file_sha256(TRAINED_WEIGHTS) == config["adapted_weights_sha256"], "Unexpected adapted weights"
parent_drive.fetch_folder("previews")

run_drive = DriveRun.open(api, ("RHUH-GBM", "synthesis"), ("EXP-2026-003", RUN_TAG), OUTPUT,
                          expect_in_base=("EXP-2026-001",), create=True)
preflight_drive = DriveRun(api, run_drive.folder_id, LOCAL_BASE / "preflight" / RUN_TAG)
preflight = preflight_drive.preflight()
print("Drive write/read/MD5 preflight passed; free GiB:", preflight["free_gib"])

## 3. Stage only the selected training/validation exports locally
The parent hash is verified before reading its selection. No test exports are copied or loaded.

In [ ]:
PRIVATE = Path(DRIVE_RHUH) / "synthesis" / "EXP-2026-001"
split = json.loads((PRIVATE / "split/split.json").read_text())
selections = {"train": parent["selection"], "val": parent["val_selection"]}
for group, entries in selections.items():
    allowed = set(split["roles"][group]["patients"])
    assert {e["patient"] for e in entries} <= allowed, "Selection crosses split"
    assert not {e["patient"] for e in entries} & set(split["roles"]["test"]["patients"])
    for entry in entries:
        patient = entry["patient"]
        assert Path(patient).name == patient and patient not in (".", "..")
        options = list(PRIVATE.glob(f"export_v1/train*/{patient}.npz")) if group == "train" else [PRIVATE / "export_v1/val" / f"{patient}.npz"]
        assert len(options) == 1 and options[0].is_file(), "Missing or ambiguous export"
        for source in (options[0], options[0].with_suffix(".json")):
            target = DATASET / source.relative_to(PRIVATE)
            target.parent.mkdir(parents=True, exist_ok=True)
            if not target.exists() or file_sha256(target) != file_sha256(source):
                shutil.copyfile(source, target)
for relative in ("split/split.json", "registration_policy/acceptance_policy_v1.json"):
    target = DATASET / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(PRIVATE / relative, target)
print("Selected exports staged locally; test set untouched")

## 4. Audit the parent and fetch the original MSF checkpoint
No VLM access or MedGemma token is required. The author checkpoint must match the frozen hash.

In [ ]:
from huggingface_hub import hf_hub_download
from rhuh_synthesis.msf_diagnostics import audit_parent_run, run_msf_diagnostic

parent_audit = audit_parent_run(PARENT)
print(json.dumps(parent_audit, indent=2))
spec = config["original_checkpoint"]
ORIGINAL_CHECKPOINT = Path(hf_hub_download(spec["model_id"], spec["filename"]))
assert file_sha256(ORIGINAL_CHECKPOINT) == spec["sha256"], "Unexpected original MSF checkpoint"

## 5. Run the frozen diagnostic
Loss-phase results describe dependence under teacher-forced denoising; they do not certify patient-preserving generation. Full-phase grids require independent review. A failed attempt keeps its files and needs a new run tag.

In [ ]:
try:
    record = run_msf_diagnostic(
        "configs/experiments/EXP-2026-003.json",
        dataset_root=DATASET, parent_run_dir=PARENT,
        trained_weights=TRAINED_WEIGHTS, original_checkpoint=ORIGINAL_CHECKPOINT,
        output_dir=OUTPUT, phase=PHASE, device="cuda",
    )
finally:
    upload = run_drive.sync()
    print("Drive upload:", upload)
print(json.dumps({k: record.get(k) for k in
                  ("experiment_id", "status", "phase", "training_steps", "seconds")}, indent=2))

## Read the channel diagnostic

`prediction_relative_change` measures sensitivity, not correctness. Positive `delta_vs_own` means the intervention increased epsilon error against this paired target, suggesting the own condition was useful in that probe. Empty ROIs remain null; inspect the valid-patient/case counts, especially for residual sentinels. These are teacher-forced diagnostics, not clinical or free-generation success rates.

In [ ]:
readout = []
for model, summary in record["loss_summaries"].items():
    for group in summary["group_summaries"]:
        if group["split"] == "val" and group["arm"] == "shuffle_both":
            readout.append({"model": model, "cohort": group["cohort"],
                            "patient_count": group["patient_count"],
                            "valid_patients_by_region": group.get("valid_patients_by_region"),
                            "valid_cases_by_region": group.get("valid_cases_by_region"),
                            "image": group["image"], "mask": group["mask"]})
print(json.dumps(readout, indent=2))

## 6. Confirm persistence
Do not delete the local originals until every result file is confirmed by Drive. To run sampling later, change PHASE to full and choose a new RUN_TAG; rerun setup. No automatic training follows this diagnostic.

In [ ]:
assert run_drive.confirm(), f"Some uploads are unconfirmed; keep this session and local files at {OUTPUT}"
print("All diagnostic files confirmed by server MD5")